# Playground Feast — plan/apply, historical features, materialize, online features (SDK e HTTP)

Pré-requisitos:
- `notebooks/01_generate_mock_data.py` já executado (Delta table em `data/offline_store/driver_stats`).
- Redis + RedisInsight no ar (`docker compose up -d` na raiz do repo).
- Para a última célula (chamada HTTP), o feature server precisa estar rodando
  em outro terminal: `uv run feast --chdir feature_repo serve` (porta 6566).

In [1]:
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from deltalake import DeltaTable
from feast import FeatureStore

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent
FEATURE_REPO_DIR = REPO_ROOT / "feature_repo"
DELTA_TABLE_PATH = REPO_ROOT / "data" / "offline_store" / "driver_stats"

sys.path.insert(0, str(FEATURE_REPO_DIR))

## Passo 1 — `feast plan` (via CLI)

Nota: `FeatureStore.plan()` no SDK puro exige montar manualmente um objeto
`RepoContents` (o mesmo trabalho que o parser da CLI já faz). Para este
playground, reproduzimos o `feast plan` chamando a CLI via `subprocess`,
que é a forma pública e estável de obter o mesmo resultado.

In [2]:
result = subprocess.run(
    ["uv", "run", "feast", "--chdir", str(FEATURE_REPO_DIR), "plan"],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

No project found in the repository. Using project name feature_store_playground defined in feature_store.yaml
Updated feature view driver_stats
	batch_source: type: BATCH_FILE
timestamp_field: "event_timestamp"
created_timestamp_column: "created_timestamp"
file_options {
  file_format {
    delta_format {
    }
  }
  uri: "C:\\Users\\USER\\mlops\\feature-store-playground\\data\\offline_store\\driver_stats"
}
data_source_class_type: "feast.infra.offline_stores.file_source.FileSource"
name: "driver_stats_source"
description: "Mock de localizaÃ§Ã£o/status de motoristas, offline store em Delta Lake local"
meta {
  created_timestamp {
    seconds: 1786899990
    nanos: 865591000
  }
  last_updated_timestamp {
    seconds: 1786899991
    nanos: 119162000
  }
}
 -> type: BATCH_FILE
timestamp_field: "event_timestamp"
created_timestamp_column: "created_timestamp"
file_options {
  file_format {
    delta_format {
    }
  }
  uri: "C:\\Users\\USER\\mlops\\feature-store-playground\\data\\offline_s

## Passo 2 — `apply` via SDK

In [3]:
from entities import driver  # noqa: E402
from feature_services import driver_activity_v1  # noqa: E402
from features import driver_stats_fv  # noqa: E402

store = FeatureStore(repo_path=str(FEATURE_REPO_DIR))
store.apply([driver, driver_stats_fv, driver_activity_v1])
print("apply concluído")

apply concluído


## Passo 3 — Historical features (point-in-time join)

Os timestamps do `entity_df` são derivados dos próprios dados mockados
(min/max de `event_timestamp` por driver), para o notebook continuar
funcionando mesmo que `01_generate_mock_data.py` seja re-executado em
outro momento. Incluímos de propósito um timestamp **antes** do primeiro
evento do driver 1009, para mostrar features nulas quando não há histórico
ainda no ponto no tempo consultado.

In [4]:
raw = DeltaTable(str(DELTA_TABLE_PATH)).to_pandas().sort_values(["driver_id", "event_timestamp"])

driver_1000_events = raw[raw["driver_id"] == 1000]["event_timestamp"]
driver_1005_events = raw[raw["driver_id"] == 1005]["event_timestamp"]
driver_1009_events = raw[raw["driver_id"] == 1009]["event_timestamp"]

entity_df = pd.DataFrame(
    {
        "driver_id": [1000, 1005, 1009, 1000],
        "event_timestamp": [
            driver_1000_events.iloc[len(driver_1000_events) // 2],  # meio do histórico do driver 1000
            driver_1005_events.iloc[-1],  # último evento conhecido do driver 1005
            driver_1009_events.iloc[0] - pd.Timedelta(hours=1),  # antes de qualquer evento -> features nulas
            datetime.now(timezone.utc),  # "agora" -> deve trazer o último valor materializado do driver 1000
        ],
    }
)

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_stats:latitude",
        "driver_stats:longitude",
        "driver_stats:status",
    ],
).to_df()
training_df

c:\Users\USER\mlops\feature-store-playground\.venv\Lib\site-packages\ibis\backends\duckdb\__init__.py:332: DeprecationWarning: fetch_arrow_table() is deprecated, use to_arrow_table() instead.
  meta = result.fetch_arrow_table()


,driver_id,event_timestamp,latitude,longitude,status
0,1000,2026-08-08 06:10:09.181619+00:00,-23.515951,-46.693542,offline
1,1005,2026-08-09 11:22:09.181619+00:00,-23.507013,-46.612980,en_route
2,1000,2026-08-16 17:26:44.770752+00:00,-23.514484,-46.686771,available
3,1009,2026-08-06 17:57:09.181619+00:00,NaN,NaN,None


## Passo 4 — Materialize para o Redis

A primeira materialização completa (cobrindo todo o histórico mockado) foi
feita via CLI em `docs/tasks/07-materialize.md`
(`feast materialize <inicio> <fim>`), necessária porque `materialize_incremental`
usa `(agora - ttl)` como início apenas quando não há checkpoint prévio.
Aqui usamos `materialize_incremental`, o padrão do dia a dia: materializa
somente o que é novo desde a última materialização registrada.

In [5]:
store.materialize_incremental(end_date=datetime.now(timezone.utc))

Materializing 1 feature views to 2026-08-16 17:26:45+00:00 into the redis online store.

driver_stats from 2026-08-16 17:10:46+00:00 to 2026-08-16 17:26:45+00:00:


c:\Users\USER\mlops\feature-store-playground\.venv\Lib\site-packages\ibis\backends\duckdb\__init__.py:332: DeprecationWarning: fetch_arrow_table() is deprecated, use to_arrow_table() instead.
  meta = result.fetch_arrow_table()


## Passo 5 — Online features via SDK

In [6]:
online_sdk = store.get_online_features(
    features=[
        "driver_stats:latitude",
        "driver_stats:longitude",
        "driver_stats:status",
    ],
    entity_rows=[{"driver_id": 1000}, {"driver_id": 1005}],
).to_dict()
online_sdk

{'driver_id': [1000, 1005],
 'longitude': [-46.686771392822266, -46.612979888916016],
 'latitude': [-23.514484405517578, -23.50701332092285],
 'status': ['available', 'en_route']}

## Passo 6 — Online features via HTTP (feature server)

Pré-requisito: em outro terminal, rodar
`uv run feast --chdir feature_repo serve` (porta padrão 6566).

In [7]:
try:
    resp = requests.post(
        # 127.0.0.1 em vez de localhost: no Windows/Docker Desktop, "localhost"
        # resolve IPv6 (::1) primeiro, que não responde nas portas publicadas
        # pelo Docker, causando ~5s de espera por chamada antes do fallback
        # para IPv4. Indo direto em 127.0.0.1 evita essa espera.
        "http://127.0.0.1:6566/get-online-features",
        json={
            "features": [
                "driver_stats:latitude",
                "driver_stats:longitude",
                "driver_stats:status",
            ],
            "entities": {"driver_id": [1000, 1005]},
        },
        timeout=15,  # a primeira chamada pode ser mais lenta (warm-up do servidor)
    )
    resp.raise_for_status()
    online_http = resp.json()
    print(online_http)
except requests.exceptions.ConnectionError:
    online_http = None
    print(
        "Feature server não respondeu em 127.0.0.1:6566 — rode "
        "`uv run feast --chdir feature_repo serve` em outro terminal e "
        "execute esta célula novamente."
    )

{'results': [{'values': [1000, 1005], 'statuses': ['PRESENT', 'PRESENT'], 'event_timestamps': ['1970-01-01T00:00:00Z', '1970-01-01T00:00:00Z']}, {'values': [-46.686771392822266, -46.612979888916016], 'statuses': ['PRESENT', 'PRESENT'], 'event_timestamps': ['2026-08-09T05:06:09.181619Z', '2026-08-09T11:22:09.181619Z']}, {'values': [-23.514484405517578, -23.50701332092285], 'statuses': ['PRESENT', 'PRESENT'], 'event_timestamps': ['2026-08-09T05:06:09.181619Z', '2026-08-09T11:22:09.181619Z']}, {'values': ['available', 'en_route'], 'statuses': ['PRESENT', 'PRESENT'], 'event_timestamps': ['2026-08-09T05:06:09.181619Z', '2026-08-09T11:22:09.181619Z']}], 'metadata': {'feature_names': ['driver_id', 'longitude', 'latitude', 'status']}}


## Passo 7 — Comparação SDK vs HTTP

Confere que os valores retornados pelas duas vias batem (mesma fonte no Redis).

In [8]:
if online_http is not None:
    print("SDK :", online_sdk["latitude"], online_sdk["longitude"], online_sdk["status"])
    print("HTTP:", online_http["results"])

SDK : [-23.514484405517578, -23.50701332092285] [-46.686771392822266, -46.612979888916016] ['available', 'en_route']
HTTP: [{'values': [1000, 1005], 'statuses': ['PRESENT', 'PRESENT'], 'event_timestamps': ['1970-01-01T00:00:00Z', '1970-01-01T00:00:00Z']}, {'values': [-46.686771392822266, -46.612979888916016], 'statuses': ['PRESENT', 'PRESENT'], 'event_timestamps': ['2026-08-09T05:06:09.181619Z', '2026-08-09T11:22:09.181619Z']}, {'values': [-23.514484405517578, -23.50701332092285], 'statuses': ['PRESENT', 'PRESENT'], 'event_timestamps': ['2026-08-09T05:06:09.181619Z', '2026-08-09T11:22:09.181619Z']}, {'values': ['available', 'en_route'], 'statuses': ['PRESENT', 'PRESENT'], 'event_timestamps': ['2026-08-09T05:06:09.181619Z', '2026-08-09T11:22:09.181619Z']}]
